In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, QuantileDiscretizer, VectorAssembler
from pyspark.ml import Pipeline
import pickle
from pathlib import Path
import json

In [0]:
# ============================================================
# Optimized spark dataframe value transformation helper
# ============================================================

def create_preprocessing_pipeline(df, cat_cols, num_cols, id_col):
    """
    Create a single MLlib pipeline for all transformations.
    This is much more efficient than sequential transformations.
    """
    stages = []
    
    # Categorical encoding using StringIndexer (MLlib)
    str_cols = cat_cols + [id_col]
    indexers = StringIndexer(inputCols=str_cols, outputCols=[c+'_idx' for c in str_cols], handleInvalid="keep")

    # Numerical normalization using QuantileDiscretizer with many bins
    # This approximates quantile normalization but is much faster
    discretizers = QuantileDiscretizer(numBuckets=1000,
                                    inputCols=num_cols,
                                    outputCols=[c + "_norm" for c in num_cols],
                                    handleInvalid="keep",
                                    relativeError=0.001  # Balance between accuracy and speed
                                    )

    # Create and fit pipeline
    pipeline = Pipeline(stages=[indexers, discretizers])

    return pipeline


def process_spark_dataframes_optimized(df, id_col, cat_cols, num_cols, emb_col, 
                                       partition_col, emb_dim, fitted_pipeline=None):
    """
    Optimized processing using MLlib pipeline.
    
    Args:
        fitted_pipeline: If provided, use this fitted pipeline (for inference)
                        If None, fit a new pipeline (for training)
    """
    # Handle nulls in embedding column first
    df = df.withColumn(
        emb_col,
        F.coalesce(
            F.col(emb_col),
            F.array_repeat(F.lit(0.0), emb_dim)
        )
    )
    
    # Apply or fit pipeline
    if fitted_pipeline is None:
        pipeline = create_preprocessing_pipeline(df, cat_cols, num_cols, id_col)
        fitted_pipeline = pipeline.fit(df)
    
    df_transformed = fitted_pipeline.transform(df)
    
    # Shift indices by 1 to reserve 0 for padding (do this in bulk)
    idx_cols = [id_col + "_idx"] + [c + "_idx" for c in cat_cols]
    for idx_col in idx_cols:
        df_transformed = df_transformed.withColumn(
            idx_col, 
            (F.col(idx_col) + 1).cast("int")
        )
    
    # Normalize the discretized values to [0, 1] range
    for col in num_cols:
        df_transformed = df_transformed.withColumn(
            col + "_norm",
            F.col(col + "_norm") / 1000.0
        )
    
    # Select only needed columns
    select_cols = [
        id_col,
        id_col + '_idx',
        *cat_cols,
        *[c + "_idx" for c in cat_cols],
        *num_cols,
        *[c + "_norm" for c in num_cols],
        emb_col,
        partition_col
    ]
    
    df_proc = df_transformed.select(*select_cols)
    
    return df_proc, fitted_pipeline


def save_metadata_optimized(user_df, post_df, user_id_col, post_id_col, 
                           user_cat_cols, post_cat_cols, user_num_cols, 
                           post_num_cols, metadata_dir, user_pipeline, post_pipeline):
    """
    Optimized metadata saving with bulk operations.
    """
    # Save pipelines directly - they contain all transformation logic
    user_pipeline.write().overwrite().save(f"{metadata_dir}/user_pipeline")
    post_pipeline.write().overwrite().save(f"{metadata_dir}/post_pipeline")
    
    # Collect unique IDs using aggregation
    known_user_ids = set([row[user_id_col] for row in 
                         user_df.select(user_id_col).distinct().toLocalIterator()])
    known_post_ids = set([row[post_id_col] for row in 
                         post_df.select(post_id_col).distinct().toLocalIterator()])
    
    # Collect categorical values
    known_user_cat_values = {}
    for col in user_cat_cols:
        known_user_cat_values[col] = set([row[col] for row in 
                                         user_df.select(col).distinct().toLocalIterator()])
    
    known_post_cat_values = {}
    for col in post_cat_cols:
        known_post_cat_values[col] = set([row[col] for row in 
                                         post_df.select(col).distinct().toLocalIterator()])
    
    # Save metadata
    Path(metadata_dir).mkdir(parents=True, exist_ok=True)
    pickle.dump(known_user_ids, open(Path(metadata_dir) / 'known_user_ids.pkl', 'wb'))
    pickle.dump(known_post_ids, open(Path(metadata_dir) / 'known_post_ids.pkl', 'wb'))
    pickle.dump(known_user_cat_values, open(Path(metadata_dir) / 'known_user_cat_values.pkl', 'wb'))
    pickle.dump(known_post_cat_values, open(Path(metadata_dir) / 'known_post_cat_values.pkl', 'wb'))


def load_spark_dataframes(spark, db, table_names, run_date, sample_window=180):
    """Load dataframes with optimized filters."""
    # Use single date calculation
    start_date = F.expr(f"date_sub('{run_date}', {sample_window})")
    
    user_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['user_features']}
        WHERE partition_date BETWEEN date_sub('{run_date}', {sample_window}) AND '{run_date}'
    """)
    
    post_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['post_features']}
        WHERE partition_date BETWEEN date_sub('{run_date}', {sample_window}) AND '{run_date}'
    """)
    
    interaction_df = spark.sql(f"""
        SELECT * 
        FROM {db}.{table_names['user_post_interaction_agg']}
        WHERE partition_date BETWEEN date_sub('{run_date}', {sample_window}) AND '{run_date}'
    """)
    
    # Cache these dataframes as they'll be used multiple times
    user_df.cache()
    post_df.cache()
    interaction_df.cache()
    
    return user_df, post_df, interaction_df


def save_spark_dataframes(spark, user_df, post_df, joined_df, partition_col, db, table_names, parquet_dir):
    # save processed feature data as delta tables
    user_df.write.format("delta") \
        .partitionBy(partition_col) \
        .mode("overwrite") \
        .saveAsTable("{db}.{user_processed_features}".format(db=db, user_processed_features=table_names['user_processed_features']))
    post_df.write.format("delta") \
        .partitionBy(partition_col) \
        .mode("overwrite") \
        .saveAsTable("{db}.{post_processed_features}".format(db=db, post_processed_features=table_names['post_processed_features']))
    
    # Repartition before writing for better parallelism
    # Write with compression for better I/O
    (joined_df.repartition(partition_col)
     .write
     .mode("overwrite")
     .option("compression", "snappy")
     .parquet(parquet_dir)) 
    
    return None


def process_user_post_spark_dataframes_optimized(
    user_df, post_df, interaction_df, 
    user_id_col, post_id_col,
    user_cat_cols, user_num_cols,
    post_cat_cols, post_num_cols,
    user_emb_col, post_emb_col,
    label_col, partition_col,
    emb_dim,
    spark,
    db,
    table_names,
    parquet_dir,
    metadata_dir
):
    """
    Optimized processing with MLlib pipelines and efficient joins.
    """
    
    # Process User Table with pipeline
    user_df_proc, user_pipeline = process_spark_dataframes_optimized(
        user_df, user_id_col, user_cat_cols, user_num_cols, 
        user_emb_col, partition_col, emb_dim
    )
    
    # Process Post Table with pipeline
    post_df_proc, post_pipeline = process_spark_dataframes_optimized(
        post_df, post_id_col, post_cat_cols, post_num_cols, 
        post_emb_col, partition_col, emb_dim
    )
    
    # Repartition
    interaction_df = interaction_df.repartition(user_id_col, post_id_col, partition_col)
    user_df_proc = user_df_proc.repartition(user_id_col, partition_col)
    post_df_proc = post_df_proc.repartition(post_id_col, partition_col)
    
    # Cache processed dataframes before joins
    user_df_proc.cache()
    post_df_proc.cache()
    
    # Optimized join: broadcast smaller table if possible
    # PySpark will auto-broadcast if one side is small enough
    joined_df = (interaction_df
                .join(user_df_proc, [user_id_col, partition_col], "left")
                .join(post_df_proc, [post_id_col, partition_col], "left"))
    
    # Convert label using when clause
    joined_df = joined_df.withColumn(
        label_col, 
        F.when(F.col("interaction") > 1, 1).otherwise(0)
    )
    
    # Save processed feature data and joined table
    save_spark_dataframes(spark, user_df_proc, post_df_proc, joined_df, partition_col, db, table_names, parquet_dir)

    # Save metadata
    save_metadata_optimized(
        user_df_proc, post_df_proc, 
        user_id_col, post_id_col, 
        user_cat_cols, post_cat_cols, 
        user_num_cols, post_num_cols, 
        metadata_dir,
        user_pipeline, post_pipeline
    )
    
    # Collect statistics efficiently using single pass aggregations
    stats_df = user_df_proc.agg(
        F.countDistinct(user_id_col).alias("n_users"),
        *[F.countDistinct(cat_col + '_idx').alias(f"user_cat_{i}") 
          for i, cat_col in enumerate(user_cat_cols)]
    )
    user_stats = stats_df.first()
    
    stats_df = post_df_proc.agg(
        F.countDistinct(post_id_col).alias("n_posts"),
        *[F.countDistinct(cat_col + '_idx').alias(f"post_cat_{i}") 
          for i, cat_col in enumerate(post_cat_cols)]
    )
    post_stats = stats_df.first()
    
    # Unpersist cached dataframes
    user_df_proc.unpersist()
    post_df_proc.unpersist()
    user_df.unpersist()
    post_df.unpersist()
    interaction_df.unpersist()
    
    return {
        'n_users': user_stats['n_users'],
        'n_posts': post_stats['n_posts'],
        'user_cat_dims': [user_stats[f'user_cat_{i}'] for i in range(len(user_cat_cols))],
        'post_cat_dims': [post_stats[f'post_cat_{i}'] for i in range(len(post_cat_cols))],
        'user_num_dim': len(user_num_cols),
        'post_num_dim': len(post_num_cols),
        'parquet_dir': parquet_dir,
        'metadata_dir': metadata_dir
    }

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
data_args = dict(
        user_id_col='user_id', 
        post_id_col='post_id',
        user_cat_cols=["ip_province", "vehicle_series", "platform"],
        user_num_cols=["months_since_registration", "months_since_consent", "identity", "is_koc", "has_profile_photo",
                "community_visits", "posts_published", "posts_viewed", "posts_liked", 
                "posts_commented", "posts_shared", "users_followed", "tribes_joined"],
        post_cat_cols=["post_type"], 
        post_num_cols=["days_since_published", "is_ugc", "is_sink", "has_pic", "has_video",
                 "views", "likes", "comments", "shares", "dwell_time"],
        user_emb_col='lastn_embs', 
        post_emb_col='post_content_emb',
        label_col='label',
        partition_col='partition_date'
)
sample_window = model_config['data_sample_window']
emb_dim = model_config['embedding_dim']

parquet_dir=model_config['TWO_TOWER_TRAIN_PARQUET_PATH']
metadata_dir=model_config['TWO_TOWER_METADATA_PATH']

In [0]:
# Load Data
user_df, post_df, interaction_df = load_spark_dataframes(
    spark, db, table_names, run_date, sample_window
)

In [0]:
# Process Data
metadata = process_user_post_spark_dataframes_optimized(
    user_df, post_df, interaction_df, 
    emb_dim=emb_dim, 
    spark=spark, 
    db=db, 
    table_names=table_names,
    parquet_dir=parquet_dir, 
    metadata_dir=metadata_dir, 
    **data_args
)

print("Processing complete!")
print(f"Number of users: {metadata['n_users']}")
print(f"Number of posts: {metadata['n_posts']}")

In [0]:
pickle.dump(metadata, open(Path(metadata_dir) / 'metadata_stats.pkl', 'wb'))